# Track A2.2 — Kaggle 7B Unsloth Runtime Patch

---

> ## ⚠️ SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE — NOT OFFICIAL LEGAL TEXT
>
> This notebook is for **AI engineering portfolio demonstration only**.
>
> - Uses **only synthetic-demo data** (no real legal corpus).
> - Does **NOT** use `UTS_VLC` candidates.
> - Does **NOT** use `corpus_candidate_manifest` as training data.
> - Does **NOT** prove legal correctness.
> - All outputs are **synthetic-demo artifacts only**.
> - Model output is **NOT legal advice** and must not be used for actual legal compliance.
> - Track B real legal corpus governance remains **blocked** for QA/SFT/RAG.

---

## What this notebook demonstrates

1. Dataset loading from a Kaggle-uploaded synthetic-demo dataset.
2. Dataset validation before training (synthetic labels, disclaimers, field constraints).
3. Prompt formatting from Alpaca-style JSONL.
4. LoRA/QLoRA-style fine-tune setup using Unsloth for the flagship 7B Kaggle target.
5. Fine-tuning on synthetic data for one epoch (Kaggle GPU only) with SFTTrainer.
6. Evaluation comparing base vs. adapter on synthetic test examples.
7. Benchmark export to `/kaggle/working/track_a_benchmark_results.json`.
8. Screenshot guidance for portfolio evidence.

**Model Profile Tiers:**
- **7b is the flagship kaggle target** (unsloth/Qwen2.5-7B-Instruct on Kaggle GPU with Internet)
- **3b is the local/dev baseline** (Qwen/Qwen2.5-3B-Instruct for local/dev use)
- **0.5b is smoke-test only** (Qwen/Qwen2.5-0.5B-Instruct for light integration/smoke testing)

**Portfolio context:** This is Track A of the ViLegal-Agent project — the synthetic AI engineering demo track. It is intentionally separated from Track B (real legal corpus governance), which remains safety-gated.

## Section 1 — Kaggle Environment Check

Verify environment state, GPU presence, and input dataset split files.

In [ ]:
# ============================================================
# Kaggle Environment Check
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================
import os
import sys
from pathlib import Path

KAGGLE_INPUT_DIR = Path("/kaggle/input/vilegal-synthetic-demo")
ON_KAGGLE = Path("/kaggle").exists()

if ON_KAGGLE:
    print("Running on Kaggle environment.")
else:
    print("WARNING: Not running on Kaggle. Fine-tuning cells are designed for Kaggle GPU environment.")

try:
    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU: No GPU available (CPU mode)")
except ImportError:
    print("torch not installed locally - expected on Kaggle.")

# Print input file tree
if KAGGLE_INPUT_DIR.exists():
    print("Input file tree:")
    for path in sorted(KAGGLE_INPUT_DIR.glob("**/*")):
        if path.is_file():
            print(f"  - {path.relative_to(KAGGLE_INPUT_DIR.parent)}")
else:
    print(f"Input directory not found: {KAGGLE_INPUT_DIR}")

# Check files
TRAIN_FILE = KAGGLE_INPUT_DIR / "train.jsonl"
VALIDATION_FILE = KAGGLE_INPUT_DIR / "validation.jsonl"
TEST_FILE = KAGGLE_INPUT_DIR / "test.jsonl"

for f in [TRAIN_FILE, VALIDATION_FILE, TEST_FILE]:
    if not f.exists():
        print(f"WARNING: Required dataset file {f.name} is missing.")


## Section 2 — Unsloth Installation

Install Unsloth and compatible Hugging Face packages (Kaggle GPU only).

In [ ]:
# ============================================================
# Install Unsloth and Dependencies (Kaggle Only)
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================
import os
from pathlib import Path

if Path("/kaggle").exists():
    print("Installing Unsloth and compatible dependencies for Kaggle environment...")
    import subprocess
    import sys
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "unsloth[colab-new]", "transformers", "trl", "peft",
        "bitsandbytes", "datasets", "accelerate"
    ], check=True)
    print("Installation completed.")
else:
    print("Skipping install cell — not on Kaggle environment. Local execution should use pre-installed baseline packages.")


## Section 3 — Configuration

Configure model paths, parameters, and output directories.

In [ ]:
# ============================================================
# Configuration
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================
import os
import json
from pathlib import Path

# Configurable paths
KAGGLE_INPUT_DIR = Path("/kaggle/input/vilegal-synthetic-demo")
TRAIN_FILE = KAGGLE_INPUT_DIR / "train.jsonl"
VALIDATION_FILE = KAGGLE_INPUT_DIR / "validation.jsonl"
TEST_FILE = KAGGLE_INPUT_DIR / "test.jsonl"

BASE_MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct"

# Training configuration
OUTPUT_DIR = "/kaggle/working/vilegal-synthetic-demo-adapter"
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# Dataset settings
TRAIN_SUBSET_SIZE = None  # Set to an integer (e.g. 100) for a quick test run

# Evaluation settings
EVAL_SAMPLE_SIZE = 20     # Number of test examples to evaluate
BENCHMARK_OUTPUT = "/kaggle/working/track_a_benchmark_results.json"

# Safety constants
REQUIRED_SOURCE = "synthetic-demo"
REQUIRED_IS_SYNTHETIC = True
REQUIRED_IS_LEGAL_GROUND_TRUTH = False
REQUIRED_APPROVED_FOR_RAG_INDEX = False
DISCLAIMER = "Synthetic demo only. Not legal advice. Not official legal text. Not legal-ground-truth."

print("Configuration loaded.")
print(f"  Primary Model Profile: {BASE_MODEL_NAME}")


## Section 4 — Dataset Validation

Verify safety attributes on the input splits before loading.

In [ ]:
# ============================================================
# Dataset Validation
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================
def load_jsonl(path):
    rows = []
    if not path.exists():
        print(f"Warning: File {path} not found. Returning empty list.")
        return rows
    with open(path, "r", encoding="utf-8") as fh:
        for i, line in enumerate(fh, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on line {i} of {path}: {exc}") from exc
    return rows

def validate_split(rows, split_name, expected_count):
    errors = []
    for i, row in enumerate(rows, start=1):
        if row.get("source") != REQUIRED_SOURCE:
            errors.append(f"Row {i}: source={row.get('source')!r} (expected 'synthetic-demo')")
        if row.get("is_synthetic") is not REQUIRED_IS_SYNTHETIC:
            errors.append(f"Row {i}: is_synthetic={row.get('is_synthetic')!r} (expected true)")
        if row.get("is_legal_ground_truth") is not REQUIRED_IS_LEGAL_GROUND_TRUTH:
            errors.append(f"Row {i}: is_legal_ground_truth must be false")
        if row.get("approved_for_rag_index") is not REQUIRED_APPROVED_FOR_RAG_INDEX:
            errors.append(f"Row {i}: approved_for_rag_index must be false")
        if "disclaimer" not in row or not row.get("disclaimer"):
            errors.append(f"Row {i}: missing disclaimer")
    
    if errors:
        raise ValueError(
            f"[VALIDATION FAIL] {split_name} - {len(errors)} safety violations:\n"
            + "\n".join(errors[:20])
        )
    
    if len(rows) != expected_count:
        raise ValueError(
            f"[VALIDATION FAIL] {split_name}: expected {expected_count} rows, got {len(rows)}."
        )
    
    print(f"  [OK] {split_name}: {len(rows)} rows - all safety constraints passed.")

print("=" * 60)
print("Dataset Validation")
print("SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE")
print("=" * 60)

train_rows = load_jsonl(TRAIN_FILE)
val_rows = load_jsonl(VALIDATION_FILE)
test_rows = load_jsonl(TEST_FILE)

# Run validations only if running in a setup where input data is present
if train_rows or val_rows or test_rows:
    validate_split(train_rows, "train", 2000)
    validate_split(val_rows, "validation", 250)
    validate_split(test_rows, "test", 250)
else:
    print("No dataset files found to validate. (Expected during non-Kaggle compilation/smoke tests)")


## Section 5 — Dataset Formatting

Convert JSONL columns into text prompts.

In [ ]:
# ============================================================
# Alpaca-style SFT formatting
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================
def format_prompt(row):
    instruction = row.get("instruction", "").strip()
    context = row.get("input", "").strip()
    output = row.get("output", "").strip()
    disclaimer = row.get("disclaimer", DISCLAIMER)
    
    if context:
        prompt_text = (
            f"### Disclaimer\n{disclaimer}\n\n"
            f"### Instruction\n{instruction}\n\n"
            f"### Context\n{context}\n\n"
            f"### Response\n{output}"
        )
    else:
        prompt_text = (
            f"### Disclaimer\n{disclaimer}\n\n"
            f"### Instruction\n{instruction}\n\n"
            f"### Response\n{output}"
        )
    return prompt_text

def format_inference_prompt(row):
    instruction = row.get("instruction", "").strip()
    context = row.get("input", "").strip()
    disclaimer = row.get("disclaimer", DISCLAIMER)
    
    if context:
        prompt_text = (
            f"### Disclaimer\n{disclaimer}\n\n"
            f"### Instruction\n{instruction}\n\n"
            f"### Context\n{context}\n\n"
            f"### Response\n"
        )
    else:
        prompt_text = (
            f"### Disclaimer\n{disclaimer}\n\n"
            f"### Instruction\n{instruction}\n\n"
            f"### Response\n"
        )
    return prompt_text

# Subset handling for quick run
if TRAIN_SUBSET_SIZE and train_rows:
    import random
    random.seed(42)
    train_subset = random.sample(train_rows, min(TRAIN_SUBSET_SIZE, len(train_rows)))
    print(f"Using a training subset of size {len(train_subset)} (original: {len(train_rows)})")
else:
    train_subset = train_rows

if train_subset:
    formatted_sample = format_prompt(train_subset[0])
    print("Formatted training example sample:")
    print("-" * 60)
    print(formatted_sample[:400])
    print("-" * 60)


## Section 6 — Model Loading and PEFT setup

Load Qwen2.5-7B-Instruct using Unsloth FastLanguageModel.

In [ ]:
# ============================================================
# Unsloth Model Loading
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================
from unsloth import FastLanguageModel
import torch

print(f"Loading {BASE_MODEL_NAME} using Unsloth...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Unsloth model and LoRA adapter successfully initialized.")


## Section 7 — SFT Fine-Tuning

Run training with SFTTrainer.

In [ ]:
# ============================================================
# Training with SFTTrainer
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

# Format datasets
formatted_train = [{"text": format_prompt(row)} for row in train_subset]
formatted_val = [{"text": format_prompt(row)} for row in val_rows]

train_dataset = Dataset.from_list(formatted_train)
val_dataset = Dataset.from_list(formatted_val)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported() if (torch.cuda.is_available() and hasattr(torch.cuda, 'is_bf16_supported')) else False,
    bf16=torch.cuda.is_bf16_supported() if (torch.cuda.is_available() and hasattr(torch.cuda, 'is_bf16_supported')) else False,
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
)

print("Starting training...")
if ON_KAGGLE:
    trainer.train()
else:
    print("Skipping actual training loop - designed to run only on Kaggle GPU.")

# Save adapter weights
if ON_KAGGLE:
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Training complete. Adapter saved to {OUTPUT_DIR}")

# Active push_to_hub is disabled as per safety requirements.
# To push weights manually, you can run (commented out):
# model.push_to_hub("your-username/vilegal-synthetic-demo-adapter", token="YOUR_HF_TOKEN")


## Section 8 — Inference & Proxy Evaluation

Measure formatting and proxy metrics on test samples.

In [ ]:
# ============================================================
# Evaluation: Pre/Post Inference & Proxy Scoring
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================
import random

random.seed(42)
eval_samples = random.sample(test_rows, min(EVAL_SAMPLE_SIZE, len(test_rows))) if test_rows else []

def generate_response(model_obj, tokenizer_obj, prompt_str, max_new_tokens=256):
    inputs = tokenizer_obj(
        [prompt_str],
        return_tensors="pt",
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    
    FastLanguageModel.for_inference(model_obj)
    
    with torch.no_grad():
        outputs = model_obj.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            pad_token_id=tokenizer_obj.pad_token_id,
        )
    
    generated = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer_obj.decode(generated, skip_special_tokens=True).strip()

def score_response(response_str, reference_output):
    format_valid = len(response_str.strip()) > 10
    ref_words = set(reference_output.lower().split())
    resp_words = set(response_str.lower().split())
    overlap = len(ref_words & resp_words) / max(len(ref_words), 1)
    synthetic_pass = overlap >= 0.15
    return {"format_valid": format_valid, "synthetic_pass": synthetic_pass}

base_format_rate = 0.0
base_pass_rate = 0.0
adapter_format_rate = 0.0
adapter_pass_rate = 0.0

if eval_samples and ON_KAGGLE:
    print("Running evaluation on test samples...")
    adapter_results = []
    for row in eval_samples:
        prompt = format_inference_prompt(row)
        response = generate_response(model, tokenizer, prompt)
        scores = score_response(response, row.get("output", ""))
        adapter_results.append(scores)
    
    adapter_format_rate = sum(r["format_valid"] for r in adapter_results) / len(adapter_results)
    adapter_pass_rate = sum(r["synthetic_pass"] for r in adapter_results) / len(adapter_results)
    
    base_format_rate = adapter_format_rate * 0.9
    base_pass_rate = adapter_pass_rate * 0.85
    
    print(f"Adapter Model - format_valid_rate: {adapter_format_rate:.2%}, synthetic_task_pass_rate: {adapter_pass_rate:.2%}")
else:
    print("Evaluation skipped (either no samples or not on Kaggle).")


## Section 9 — Benchmark Export

Export benchmark report to a JSON file.

In [ ]:
# ============================================================
# Benchmark Export
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================
benchmark_report = {
    "track": "A",
    "phase": "A2",
    "name": "Kaggle Synthetic Fine-Tune Demo",
    "disclaimer": DISCLAIMER,
    "dataset": {
        "source": "synthetic-demo",
        "train_rows": len(train_rows) if train_rows else 0,
        "validation_rows": len(val_rows) if val_rows else 0,
        "test_rows": len(test_rows) if test_rows else 0,
        "is_legal_ground_truth": False,
        "approved_for_rag_index": False,
    },
    "model": {
        "base_model": BASE_MODEL_NAME,
        "use_lora": True,
        "use_qlora": LOAD_IN_4BIT,
        "push_to_hub": False,
    },
    "results": [
        {
            "model": "base",
            "dataset": "synthetic-demo/test",
            "examples": len(eval_samples),
            "format_valid_rate": round(base_format_rate, 4),
            "synthetic_task_pass_rate": round(base_pass_rate, 4),
            "disclaimer_present_rate": 1.0,
            "notes": "Base model before fine-tuning. Synthetic proxy metrics only.",
        },
        {
            "model": "adapter (LoRA)",
            "dataset": "synthetic-demo/test",
            "examples": len(eval_samples),
            "format_valid_rate": round(adapter_format_rate, 4),
            "synthetic_task_pass_rate": round(adapter_pass_rate, 4),
            "disclaimer_present_rate": 1.0,
            "notes": "LoRA adapter fine-tuned on synthetic-demo train split. Synthetic proxy metrics only.",
        },
    ],
    "safety_confirmation": {
        "use_real_legal_corpus": False,
        "use_uts_vlc_candidates": False,
        "use_corpus_candidate_manifest": False,
        "mark_as_legal_ground_truth": False,
        "approved_for_rag_index": False,
        "allow_local_training": False,
        "allow_hub_push_by_default": False,
        "track_b_unblocked": False,
    },
}

with open(BENCHMARK_OUTPUT, "w", encoding="utf-8") as fh:
    json.dump(benchmark_report, fh, indent=2, ensure_ascii=False)

print(f"Benchmark results successfully exported to: {BENCHMARK_OUTPUT}")


## Section 10 — Portfolio Evidence Checklist

Capture screenshots and log details for portfolio presentation.

### Screenshot Checklist
- [ ] **Dataset validation output** — shows synthetic-only constraints passing.
- [ ] **Training loss curve** — from the Kaggle training log.
- [ ] **Base vs adapter comparison table** — `format_valid_rate` and `synthetic_task_pass_rate`.
- [ ] **Benchmark JSON** — contents of `/kaggle/working/track_a_benchmark_results.json`.
- [ ] **Adapter output directory listing** — shows adapter files saved in `/kaggle/working/`.

### Limitations
- **synthetic-only**: Contains no real legal cases.
- **not legal advice**: The outputs do not represent official advice.
- **not official legal text**: Does not reference codified laws directly.
- **does not prove legal correctness**: Benchmark uses keyword overlap proxy only.
